# What Is Feature Engineering?

**Topic:** Feature Engineering

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, FloatSlider, Output, HBox, VBox
from IPython.display import display, clear_output, HTML
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from tkh_utils import PALETTE, FONT, base_layout

TIMES_SQUARE_LAT, TIMES_SQUARE_LON = 40.7580, -73.9855


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the three pillars of feature engineering: creating, transforming, and selecting features
- **Explain** why raw features often fail to capture the patterns a model needs to learn
- **Interpret** a before-and-after model performance comparison and explain why the improvement came from representation, not from adding columns

> **Tip:** In the real-world example below, compare the R² score and the typical dollar error before and after feature engineering. The improvement doesn't come from adding more columns — it comes from representing the same two columns correctly.

---
## How we got here

In **[preprocessing/09_pipelines.ipynb](../preprocessing/09_pipelines.ipynb)** you learned to assemble transformations into reproducible workflows. In **[ml_concepts/10_feature_importance.ipynb](../ml_concepts/10_feature_importance.ipynb)** you learned which features a model relies on most.

Now comes a natural next question: instead of measuring which raw features matter, what if you *created better features* from scratch? That is feature engineering, and it is often the highest-leverage step in the entire machine learning workflow.

---
## Why this matters for data science

Raw data rarely arrives in the ideal form for a model to learn from. A date column is useless until you extract recency or season. A price column misleads a linear model until you log-transform it. Two columns individually tell you nothing about Manhattan private rooms until you create an interaction feature.

Think of it like a chef transforming raw ingredients. A diner cannot eat raw flour, uncracked eggs, or unmixed sugar. The chef transforms those raw materials into something the diner can enjoy. A model cannot learn from a raw date string, a skewed price column, or a free-text listing name — but it can learn from the features you engineer from them.

---
## Try it yourself

In [2]:
_airbnb_w1 = pd.read_csv('../../data/nyc_airbnb.csv')
_airbnb_w1 = _airbnb_w1[_airbnb_w1['price'] > 0].copy()
_airbnb_w1['dist_midtown'] = np.sqrt(
    (_airbnb_w1['latitude']  - TIMES_SQUARE_LAT)**2 +
    (_airbnb_w1['longitude'] - TIMES_SQUARE_LON)**2
)
_last_review_max_w1 = pd.to_datetime(_airbnb_w1['last_review']).max()
_row_w1 = _airbnb_w1.iloc[0]

_STEPS_W1 = [
    {
        "raw_col": "price", "raw_val": lambda r: f"${r['price']:.0f}",
        "eng_name": "log_price", "eng_val": lambda r: f"{np.log1p(r['price']):.3f}",
        "why": "Price is right-skewed — a few $1000+ listings stretch the scale. "
               "log1p compresses that tail so linear models see a roughly normal distribution.",
    },
    {
        "raw_col": "minimum_nights", "raw_val": lambda r: f"{r['minimum_nights']:.0f} nights",
        "eng_name": "log_min_nights", "eng_val": lambda r: f"{np.log1p(r['minimum_nights']):.3f}",
        "why": "The gap between a 1-night and 3-night minimum matters more than the gap "
               "between a 60-night and 62-night minimum — log compresses the large values.",
    },
    {
        "raw_col": "latitude & longitude", "raw_val": lambda r: f"({r['latitude']:.4f}, {r['longitude']:.4f})",
        "eng_name": "dist_midtown", "eng_val": lambda r: f"{r['dist_midtown']:.3f}",
        "why": "Neither coordinate predicts price well on its own. Combined into distance from "
               "Times Square, they capture the thing that actually drives price: how central "
               "the listing is. This one new column ends up beating both columns it was built from.",
    },
    {
        "raw_col": "number_of_reviews", "raw_val": lambda r: f"{r['number_of_reviews']:.0f} reviews",
        "eng_name": "reviews_log", "eng_val": lambda r: f"{np.log1p(r['number_of_reviews']):.3f}",
        "why": "Review counts are also right-skewed — most listings have few reviews, a handful "
               "have hundreds. Log-scaling puts them on a comparable footing.",
    },
    {
        "raw_col": "last_review", "raw_val": lambda r: f"{r['last_review']}",
        "eng_name": "days_since_last_review", "eng_val": lambda r: (
            f"{(_last_review_max_w1 - pd.to_datetime(r['last_review'])).days}"
            if pd.notna(r['last_review']) else "N/A"
        ),
        "why": "A date string is not numeric. Converting it to 'days since' gives the model a "
               "directly usable number that captures how recently the listing was active.",
    },
]

out_step = Output()

step_sl = IntSlider(value=1, min=1, max=len(_STEPS_W1), step=1,
    description="Step:", style={"description_width": "60px"},
    layout=widgets.Layout(width="380px"))

def render_step(change=None):
    s = _STEPS_W1[step_sl.value - 1]
    html = (
        f'<div style="font-family:{FONT["family"]};font-size:13px;'
        f'background:#f8f9fa;padding:14px 18px;border-radius:6px;line-height:1.6;">'
        f'<p style="margin:0 0 8px 0;"><b>Step {step_sl.value}/{len(_STEPS_W1)}</b></p>'
        f'<p style="margin:0 0 4px 0;">Raw: <code>{s["raw_col"]}</code> = {s["raw_val"](_row_w1)}</p>'
        f'<p style="margin:0 0 8px 0;">Engineered: <code>{s["eng_name"]}</code> = {s["eng_val"](_row_w1)}</p>'
        f'<p style="margin:0;color:#2F9E44;"><b>Why this helps:</b> {s["why"]}</p>'
        f'</div>'
    )
    with out_step:
        clear_output(wait=True)
        display(HTML(html))

step_sl.observe(render_step, names="value")
display(VBox([step_sl, out_step]))
render_step()

---
## What's happening?

Feature engineering systematically transforms raw data into representations that help a model learn more effectively. There are three main approaches.

**Create new features** by combining or transforming existing columns. On their own, `latitude` and `longitude` barely correlate with price. Combined into `dist_midtown` — the distance from the center of Manhattan — they produce a stronger predictor than either raw column alone.

**Transform existing features** to match what the model expects. The `price` column in Airbnb data is right-skewed: a handful of luxury listings at \$2,000+/night stretch the scale. Taking `log(price + 1)` compresses the range and makes the distribution closer to normal, which linear models strongly prefer.

**Select the best features** to reduce noise. Not every engineered feature adds signal — some add variance instead. Feature selection keeps only the ones that genuinely help.

How you write a feature down can matter more than how many features you have. The real-world example below feeds a model the exact same two columns twice — the only thing that changes is how they're represented — and the typical price guess improves anyway.

| Raw feature | Engineered feature | What it captures that raw didn't | ML benefit |
|-------------|-------------------|----------------------------------|------------|
| `latitude` + `longitude` | `dist_midtown` | Distance from the center of Manhattan | Beats either raw column alone (\|r\| = 0.35 vs 0.19 / 0.25) |
| `minimum_nights` | `log_min_nights` | Diminishing difference at long stays | Compresses right tail |
| `room_type` + `neighbourhood_group` | interaction term | Manhattan private ≠ Bronx private | Joint effect neither column has alone |
| `last_review` (date) | `days_since_last_review` | How recently the listing was active | Numeric, directly meaningful |
| `name` (text) | `word_count` | Listing description effort | Extracts signal from text |

---
## Real-world example: Airbnb Price Prediction — Before and After

The chart below trains a Ridge regression model on the NYC Airbnb dataset twice, using the exact same two categorical columns both times: `room_type` and `neighbourhood_group`. The only difference is how those columns are represented.

**Baseline** treats the categories the way a naive analyst might: alphabetically-coded integers, alongside the three raw numeric columns. **Engineered** one-hot encodes those same two categories and adds one new feature, `dist_midtown`. Both are measured with 5-fold cross-validation (shuffled) on log-transformed price.

In [3]:
airbnb = pd.read_csv('../../data/nyc_airbnb.csv')
airbnb = airbnb[airbnb['price'] > 0].copy()

airbnb['dist_midtown'] = np.sqrt(
    (airbnb['latitude']  - TIMES_SQUARE_LAT)**2 +
    (airbnb['longitude'] - TIMES_SQUARE_LON)**2
)

y = np.log1p(airbnb['price'])

# Baseline: naive alphabetically-coded integers for the two categoricals, plus raw numerics
oe = OrdinalEncoder()
airbnb[['room_type_code', 'ng_code']] = oe.fit_transform(airbnb[['room_type', 'neighbourhood_group']])
base_cols = ['minimum_nights', 'number_of_reviews', 'availability_365', 'room_type_code', 'ng_code']
X_base = airbnb[base_cols]

# Engineered: one-hot the same two categoricals, add distance from midtown
eng_cols = ['room_type', 'neighbourhood_group', 'dist_midtown']
X_eng = airbnb[eng_cols]

baseline_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('ridge', Ridge()),
])

engineered_pipe = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(), ['room_type', 'neighbourhood_group']),
        ('num', StandardScaler(), ['dist_midtown']),
    ])),
    ('ridge', Ridge()),
])

cv = KFold(5, shuffle=True, random_state=42)
r2_base = cross_val_score(baseline_pipe, X_base, y, cv=cv, scoring='r2').mean()
r2_eng = cross_val_score(engineered_pipe, X_eng, y, cv=cv, scoring='r2').mean()

# Typical dollar error on a held-out quarter of the data
idx_train, idx_test = train_test_split(airbnb.index, test_size=0.25, random_state=42)

baseline_pipe.fit(X_base.loc[idx_train], y.loc[idx_train])
engineered_pipe.fit(X_eng.loc[idx_train], y.loc[idx_train])

true_dollars = np.expm1(y.loc[idx_test])
pred_base_dollars = np.expm1(baseline_pipe.predict(X_base.loc[idx_test]))
pred_eng_dollars = np.expm1(engineered_pipe.predict(X_eng.loc[idx_test]))

mae_base = np.median(np.abs(true_dollars - pred_base_dollars))
mae_eng = np.median(np.abs(true_dollars - pred_eng_dollars))

fig = go.Figure([go.Bar(
    x=['Baseline<br>(alphabetically-coded)', 'Engineered<br>(one-hot + distance)'],
    y=[r2_base, r2_eng],
    marker_color=[PALETTE['muted'], PALETTE['primary']],
    text=[f'R² = {r2_base:.3f}<br>typical miss: ${mae_base:.0f}',
          f'R² = {r2_eng:.3f}<br>typical miss: ${mae_eng:.0f}'],
    textposition='outside',
)])
layout = base_layout(
    title='Same Two Columns, Represented Differently',
    xaxis_title='Feature set',
    yaxis_title='R² (5-fold cross-validation)',
)
layout.update(yaxis=dict(range=[0, max(r2_base, r2_eng) * 1.35]))
fig = go.Figure(data=fig.data, layout=layout)
fig.show()

print(f"Baseline:   R² = {r2_base:.3f}, typical listing is off by ${mae_base:.2f}")
print(f"Engineered: R² = {r2_eng:.3f}, typical listing is off by ${mae_eng:.2f}")
print(f"Same columns, same model — representation alone improves the typical "
      f"guess by ${mae_base - mae_eng:.2f}.")

Baseline:   R² = 0.251, typical listing is off by $50.91
Engineered: R² = 0.389, typical listing is off by $44.67
Same columns, same model — representation alone improves the typical guess by $6.24.


> **Discussion question:** Both models see the same three raw numeric columns and the same two categorical columns — only the encoding of those two categoricals changed. Why does fixing the representation of two existing columns move R² by 0.14, when simply adding more columns often helps far less?

---
## Key takeaway

> **Feature engineering is the art of giving a model the information it needs in a form it can actually use — and it often matters more than which algorithm you choose.**

---
*Next up: 02_creating_new_features — where you build ratio, difference, and combination features from Airbnb data*